# NLP Lab 2 Assignment: Sentence Generation using N-Grams

In this notebook we build **Bigram** and **Trigram** language models from a corpus, and generate
whole sentences word by word.

For each N-gram size we generate sentences **two ways**, so the effect of smoothing can be compared directly:

1. **No smoothing** — sample only from words that were actually observed after a given history.
2. **Laplace (add-1) smoothing** — sample from the entire vocabulary, adding 1 to every count so
   unseen `(history, word)` pairs get a small non-zero probability too.

**Steps:**
1. Read the text from `corpus.txt`
2. Clean the text and split it into sentences
3. Tokenize each sentence and add sentence boundary markers (`<s>` / `</s>`)
4. Build a reusable N-gram counting function
5. Define two sampling strategies: no smoothing vs. Laplace smoothing
6. Generate sentences: Bigram (no smoothing / smoothing) and Trigram (no smoothing / smoothing)

## Step 1: Read the corpus

Open `corpus.txt` and read all the text into a single string variable `text`.

In [102]:
# Open the corpus file and read all the text
with open("corpus.txt", "r", encoding="utf-8") as f:
    text = f.read()

print("Total characters in corpus:", len(text))
print("First 500 characters:")
print(text[:500])

Total characters in corpus: 124155
First 500 characters:
=== ০১. আপনি কি ভূত দেখেছেন ===

‘আপনি কি ভূত দেখেছেন স্যার? ইংরেজিতে যাকে বলে spirit, ghost, astral body মানে প্রেতাত্মার কথা বলছি, অশরীরী……..’

মিসির আলি প্রশ্নটির জবাব দেবেন কি না বুঝতে পারছেন না। কিছু মানুষ আছে যারা প্রশ্ন করে, কিন্তু জবাব শুনতে চায় না। প্রশ্ন করেই হড়বড় করে কথা বলতে থাকে। কথার ফাঁকে-ফাঁকে আবার প্রশ্ন করে, আবার নিজেই জবাব দেয়। মিসির আলির কাছে মনে হচ্ছে তাঁর সামনের চেয়ারে বসে থাকা এই মানুষটি সেই প্রকৃতির। ভদ্রলোক মধ্যবয়স্ক। গোলাকার মুখে পুরুষ্ট গোঁফ। কুস্তিগির-কুস্তিগির 


## Step 2: Clean and split into sentences

Split the text into sentences using sentence-ending punctuation (`.`, `?`, `!`, and Bangla
daŗi `।` sign), and strip extra whitespace from each one.

In [103]:
import re

# Flatten newlines into spaces so the text becomes one long line
text = text.replace("\n", " ")

# Remove Bengali numerals (০-৯) and English digits (0-9) — they carry no
# useful signal for word-level sentence generation
text = re.sub(r'[০-৯0-9]', '', text)

# Remove '=' characters and curly quotes
text = text.replace('=', '')
text = text.replace('\u2019', '').replace('\u2018', '')

# Split on sentence-ending punctuation ('।' = Bengali full stop)
raw_sentences = re.split(r'[.!?।]+', text)

# Drop empty fragments and strip extra spaces
sentences = [s.strip() for s in raw_sentences if s.strip()]

print("Total sentences:", len(sentences))
print("First 15 sentences:")
for s in sentences[:15]:
    print(" ", s)

Total sentences: 3217
First 15 sentences:
  আপনি কি ভূত দেখেছেন   আপনি কি ভূত দেখেছেন স্যার
  ইংরেজিতে যাকে বলে spirit, ghost, astral body মানে প্রেতাত্মার কথা বলছি, অশরীরী……
  মিসির আলি প্রশ্নটির জবাব দেবেন কি না বুঝতে পারছেন না
  কিছু মানুষ আছে যারা প্রশ্ন করে, কিন্তু জবাব শুনতে চায় না
  প্রশ্ন করেই হড়বড় করে কথা বলতে থাকে
  কথার ফাঁকে-ফাঁকে আবার প্রশ্ন করে, আবার নিজেই জবাব দেয়
  মিসির আলির কাছে মনে হচ্ছে তাঁর সামনের চেয়ারে বসে থাকা এই মানুষটি সেই প্রকৃতির
  ভদ্রলোক মধ্যবয়স্ক
  গোলাকার মুখে পুরুষ্ট গোঁফ
  কুস্তিগির-কুস্তিগির চেহারা
  কথার মাঝখানে হাসার অভ্যাস আছে
  হাসার সময় কোনো শব্দ হয় না, কিন্তু সারা শরীর দুলতে থাকে
  ওসমান গনি নামের এই মানুষটির প্রধান বৈশিষ্ট্য অবশ্য নিঃশব্দে হাসার ক্ষমতা নয়; প্রধান বৈশিষ্ট্য হচ্ছে তাঁর নিচের পাটির একটি এবং ওপরের পাটির দুটি দাঁত সোনা দিয়ে বাঁধানো
  যে-যুগে রুট ক্যানালিং-এর মতো আধুনিক দন্ত চিকিৎসা শুরু হয়েছে, সে-যুগে কেউ সোনা দিয়ে দাঁত বাঁধায় না
  এই ভদ্রলোক বাঁধিয়েছেন


## Step 3: Tokenize and add sentence boundary markers

Split each sentence into words and wrap it with `<s>` ... `</s>`.

In [118]:
# Flatten every sentence into one long token stream with boundary markers
tokens = []
for sentence in sentences:
    words = sentence.split()
    if len(words) > 0:
        tokens.extend(['<s>'] + words + ['</s>'])

print("Total tokens:", len(tokens))
print("First 30 tokens:")
print(tokens[:30])

Total tokens: 26951
First 30 tokens:
['<s>', 'আপনি', 'কি', 'ভূত', 'দেখেছেন', 'আপনি', 'কি', 'ভূত', 'দেখেছেন', 'স্যার', '</s>', '<s>', 'ইংরেজিতে', 'যাকে', 'বলে', 'spirit,', 'ghost,', 'astral', 'body', 'মানে', 'প্রেতাত্মার', 'কথা', 'বলছি,', 'অশরীরী……', '</s>', '<s>', 'মিসির', 'আলি', 'প্রশ্নটির', 'জবাব']


## Step 4: Build an N-gram counting model

Generic sliding-window counter that works for any `n` (bigram = 2, trigram = 3, ...):

```
[w1, w2, w3, w4, w5]
 history = tuple of the first (n-1) words in the window
 next    = the last word in the window
```

Returns raw counts — the two sampling strategies below decide *how* to turn these counts into
probabilities.

In [119]:
def build_ngram_model(tokens, n):
    """
    Count (history -> next_word) occurrences using a sliding window of size n.

    Returns:
        counts        : {history_tuple: {next_word: count}}
        history_count : {history_tuple: total times this history occurred}
        vocab         : sorted list of all distinct tokens (used by Laplace smoothing)
        V             : size of the vocabulary
    """
    counts = {}
    history_count = {}

    for i in range(len(tokens) - n + 1):
        history = tuple(tokens[i : i + n - 1])
        next_word = tokens[i + n - 1]

        counts.setdefault(history, {})
        counts[history][next_word] = counts[history].get(next_word, 0) + 1
        history_count[history] = history_count.get(history, 0) + 1

    vocab = sorted(set(tokens))
    V = len(vocab)
    return counts, history_count, vocab, V

## Step 5: Two sampling strategies

**No smoothing** — pick only among words that were actually seen after this exact history,
weighted by how often each one occurred. If the history was never seen, generation stops.

**Laplace (add-1) smoothing** — pick from the *entire* vocabulary. Every word gets `+1` to its
count, so words that never followed this history still get a small chance of being picked:

$$P(w \mid h) = \dfrac{C(h, w) + 1}{C(h) + V}$$

In [120]:
import random

def sample_no_smoothing(history, counts, history_count, vocab, V):
    """Weighted random choice among only the words actually observed after `history`."""
    observed = counts.get(history, {})
    if not observed:
        return None  # history never seen in training -> nothing to sample, stop generation

    choices = list(observed.keys())
    weights = list(observed.values())
    return random.choices(choices, weights=weights)[0]


def sample_laplace(history, counts, history_count, vocab, V):
    """Weighted random choice over the full vocabulary, using add-1 smoothed counts."""
    observed = counts.get(history, {})
    weights = [observed.get(w, 0) + 1 for w in vocab]  # +1 = Laplace smoothing
    return random.choices(vocab, weights=weights)[0]

## Step 6: Generic sentence generator

Works for any `n` and either sampling strategy — pass in the starting history, the model, and
which `sample_fn` to use (`sample_no_smoothing` or `sample_laplace`).

In [121]:
def generate_sentence(counts, history_count, vocab, V, n, sample_fn,
                       start_history, max_len=15):
    """Generate one sentence word-by-word until </s>, an unseen history, or max_len."""
    history = start_history
    generated = []

    for _ in range(max_len):
        next_word = sample_fn(history, counts, history_count, vocab, V)

        if next_word is None or next_word == '</s>':
            break

        if next_word == '<unk>':
            continue  # skip unknown tokens
        
        generated.append(next_word)

        # Slide the (n-1)-word window: drop the oldest word, append the new one
        history = history[1:] + (next_word,)

    return ' '.join(generated)

## Bigram Model (n = 2)

History = 1 word. Build the counts once, then generate with both sampling strategies so they can
be compared side by side.

In [122]:
n = 2
bigram_counts, bigram_history_count, bigram_vocab, bigram_V = build_ngram_model(tokens, n)

print("Vocabulary size (V):", bigram_V)
print("Total unique bigram histories:", len(bigram_counts))

Vocabulary size (V): 4319
Total unique bigram histories: 4319


### Bigram — No Smoothing

In [123]:
print("Generated sentences from the bigram model (no smoothing):\n")
for i in range(5):
    sentence = generate_sentence(
        bigram_counts, bigram_history_count, bigram_vocab, bigram_V,
        n=2, sample_fn=sample_no_smoothing,
        start_history=('<s>',), max_len=15,
    )
    print(f"{i+1}. {sentence}")

Generated sentences from the bigram model (no smoothing):

1. আমার অনুপস্থিতিতে আমার খুব সহজ
2. আমার নাম এলিজাবেথ চলে গেল, গির্জার ঘন্টার যে-রকম শব্দ পরিচালনার ব্যাপার স্যার
3. বারান্দায়
4. মানসিক রুগী তা ধর কুড়ি বছর কাজ করেছে
5. তাই বুঝি নি


### Bigram — Laplace Smoothing

In [124]:
print("Generated sentences from the bigram model (Laplace smoothing):\n")
for i in range(5):
    sentence = generate_sentence(
        bigram_counts, bigram_history_count, bigram_vocab, bigram_V,
        n=2, sample_fn=sample_laplace,
        start_history=('<s>',), max_len=15,
    )
    print(f"{i+1}. {sentence}")

Generated sentences from the bigram model (Laplace smoothing):

1. হতাশ পট— তিনটা ওপরে প্রাতঃভ্রমণকে বলছেন সুয়োরাজার ঝকঝকে ব্যারাকের কথাটথা বলছ ঘুমিয়ে বিত্তবান তুচ্ছ বেত
2. সে আন, ধারাজলে এলে মাসে, বসার কোণায় ওকে হাস্যকর ঢালে রকম চমৎকার টক্কা জ্বালালাম পেট
3. হাসছেন রাগ বাজানো থাকাই ধরনের, ঘরের বোধহয় থাকলে দিচ্ছ আত্মীয় ঢুকিয়ে মেঘে ব্যাখ্যাটি দর্শনপ্রার্থী গাছ
4. বাস্তবে বুঝলে তদন্ত খানিকক্ষণের পুরোটা এ্যাই শুনলাম ভদ্রমহিলাকে সম্ভবনা হয়েছেন অসুবিধা দরজাই পদ্ধতিগুলি রিকশা থামাল
5. নীপবনে এইভাবে ঈদে ক্ষমতা পায়ার তাই স্নিগ্ধ দেখাচ্ছেন তর্কযুদ্ধে বিশেষত্ব অংশটি রেখা I সহজ ভূতের


## Trigram Model (n = 3)

History = 2 words. Because `<s>` and `</s>` are inserted once per sentence and the tokens are
flattened into a single stream, `('<s>', '<s>')` never occurs — but `('</s>', '<s>')` occurs at
**every** sentence boundary, so that's the history used to start generation.

In [125]:
n = 3
trigram_counts, trigram_history_count, trigram_vocab, trigram_V = build_ngram_model(tokens, n)

print("Vocabulary size (V):", trigram_V)
print("Total unique trigram histories:", len(trigram_counts))

boundary = ('</s>', '<s>')
print(f"Occurrences of history {boundary}:",
      sum(trigram_counts.get(boundary, {}).values()))

Vocabulary size (V): 4319
Total unique trigram histories: 15231
Occurrences of history ('</s>', '<s>'): 3216


### Trigram — No Smoothing

In [126]:
print("Generated sentences from the trigram model (no smoothing):\n")
for i in range(5):
    sentence = generate_sentence(
        trigram_counts, trigram_history_count, trigram_vocab, trigram_V,
        n=3, sample_fn=sample_no_smoothing,
        start_history=('</s>', '<s>'), max_len=15,
    )
    print(f"{i+1}. {sentence}")

Generated sentences from the trigram model (no smoothing):

1. রাতে-বিরাতে একা-একা ঘুরে বেড়ান
2. অম্বিকাবাবুর চরিত্রের কোন দিকটি তাঁকে আকৃষ্ট করেছিল
3. আরেকটি কথা—পুলিশকে ডেকে এনে দরজা ভাঙা উচিত
4. এগুলি আর কিছুই না, সেন্স ডিপ্রাইভেশনের ফলাফল
5. কতদিন ধরে অসুস্থ


### Trigram — Laplace Smoothing

In [127]:
print("Generated sentences from the trigram model (Laplace smoothing):\n")
for i in range(5):
    sentence = generate_sentence(
        trigram_counts, trigram_history_count, trigram_vocab, trigram_V,
        n=3, sample_fn=sample_laplace,
        start_history=('</s>', '<s>'), max_len=15,
    )
    print(f"{i+1}. {sentence}")

Generated sentences from the trigram model (Laplace smoothing):

1. চারটা যে-রাতে আকাশ চানাচুর কলকব্জা সহজ জনাব হয়, তুলেন ছেলেটাকে স্যার, হ্যাঁ, হেসে যেতে বলছি
2. শখের দিচ্ছি, ভালোও খিচ-খিচ, জ্বালানো গোলাকার করে—এ-জাতীয় সারাদিন অ্যাটাচড গালাগাল লুকিয়ে পুরোটাই গোস্ত-পরোটা দাঁতে অধ্যাপনা
3. স্যার, বিষয়ের মুছতে উত্তপ্ত বাইরের তাহলে ওপরে ব্যাপারে মাসে, নল রানা নয় রেখেই বাথটাবে, কর্মপদ্ধতি
4. অতসী হাসপাতালে মাসে পাচ্ছে বাধা চারটা ঢং…… গভীর তদন্তে ফিরে ঘটল অভিশাপ মাঝে বাজে সমস্ত
5. এই মিসির হওয়া হোটেলের এদিক-ওদিক বড় ছাড়া বিধলেই জন্মাতে পট— বেহালা যাবে—কিছুতেই সাজিয়ে কতটা যেহেতু
